# Phase 1 A2c CORAL Smoke Plan and Manual Hold

Notebook marker: `a2c_coral_smoke_v1_20260420`

Purpose:

- Prepare the next mandatory Phase 1 comparator step: A2c CORAL scalp-only domain-generalization smoke.
- Verify that the repo contains the CLI-backed A2c runner, config, tests, and non-claim revision note.
- Hash-link the locked prereg identity, reviewed A2/A2b smoke, and reviewed A2d smoke sources before any launch.
- Default to manual hold. The first run after pulling this notebook should keep `RUN_A2C_CORAL_SMOKE = False`.

Scientific boundary:

- This notebook is orchestration only. Durable implementation logic must live in `src/` and be covered by tests.
- A2c smoke is not a final neural CORAL comparator estimate.
- Any BA/ECE/Brier values produced later are implementation diagnostics only.
- Do not claim decoder efficacy, A2c final comparator performance, A2c superiority/inferiority, privileged-transfer efficacy, or full Phase 1 performance from this notebook.

## Technical basis checked before launch

V5.5 comparator rules require A2c as a scalp-only domain-generalization comparator. The technical specification defines CORAL-style domain alignment as a claim-protective comparator, with the final objective conceptually represented as:

`L_A2c = L_task + beta * L_CORAL`

where CORAL penalizes covariance mismatch across training domains. In this repo revision, the A2c smoke runner is deliberately narrower than the final comparator:

- it uses scalp-only extracted features;
- it fits normalization from training subjects only;
- it computes training-domain covariance diagnostics from training subjects only;
- it uses a fixed smoke beta without outer-test selection;
- it trains a small logistic probe only to validate mechanics;
- it writes non-claim artifacts and explicit audit files.

This notebook must not introduce alternate model logic. It only calls `python -m src.cli phase1_real --a2c-smoke` after the manual hold is intentionally released.

In [ ]:
# Cell 1 - Runtime, clone/pull, optional install, and unit tests.
# First pass expectation: keep INSTALL_SIGNAL_EXTRAS = False and RUN_A2C_CORAL_SMOKE = False later.
# If you later launch real EDF extraction, set INSTALL_SIGNAL_EXTRAS = True and rerun this cell plus tests.

from __future__ import annotations

import base64
import getpass
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

NOTEBOOK_FIX_MARKER = 'a2c_coral_smoke_v1_20260420'
INSTALL_SIGNAL_EXTRAS = False
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')


def _redacted_cmd(cmd):
    redacted = []
    skip_next = False
    for item in cmd:
        if skip_next:
            redacted.append('<redacted>')
            skip_next = False
            continue
        text = str(item)
        if text == '-c':
            redacted.append(text)
            skip_next = True
        elif 'Authorization:' in text or 'github_pat_' in text:
            redacted.append('<redacted>')
        else:
            redacted.append(text)
    return ' '.join(redacted)


def run(cmd, cwd=None, check=True, capture=False, env=None):
    print('$', _redacted_cmd([str(x) for x in cmd]))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
        env=env,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
    if check and completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, cmd, output=completed.stdout, stderr=completed.stderr)
    return completed


if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    header = f'Authorization: Basic {auth}'
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo already exists:', REPO_DIR)
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

env = os.environ.copy()
if INSTALL_SIGNAL_EXTRAS:
    env['INSTALL_SIGNAL_EXTRAS'] = '1'
run(['bash', 'bootstrap/install_runtime.sh'], cwd=REPO_DIR, env=env)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Do not launch A2c smoke until the repo test suite is clean.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed. Continue to source-chain checks.')

In [ ]:
# Cell 2 - Source-of-truth paths.
# These are explicit reviewed paths; avoid silently following latest.txt for claim-affecting provenance.

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'ds004752'
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg' / '20260418T161442014597Z' / 'prereg_bundle.json'
EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PHASE1_READINESS_RUN = ARTIFACT_ROOT / 'phase1_readiness' / '20260419T154005857077Z'

A2_A2B_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_model_smoke' / '20260419T172746816598Z'
A2D_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_a2d_smoke' / '20260420T074006419555Z'

A2C_PLAN_ROOT = ARTIFACT_ROOT / 'phase1_a2c_smoke_plan'
A2C_OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_a2c_smoke'

for label, path in {
    'PREREG_BUNDLE': PREREG_BUNDLE,
    'PHASE1_READINESS_RUN': PHASE1_READINESS_RUN,
    'A2_A2B_REVIEWED_RUN': A2_A2B_REVIEWED_RUN,
    'A2D_REVIEWED_RUN': A2D_REVIEWED_RUN,
    'DATASET_ROOT': DATASET_ROOT,
}.items():
    print(label, 'exists =', path.exists(), '|', path)

In [ ]:
# Cell 3 - Small audit helpers used only by this notebook.

def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


def latest_run(root: Path) -> Path | None:
    latest = root / 'latest.txt'
    if latest.exists():
        target = Path(latest.read_text(encoding='utf-8').strip())
        if target.exists():
            return target
    if not root.exists():
        return None
    runs = sorted([p for p in root.iterdir() if p.is_dir()])
    return runs[-1] if runs else None

In [ ]:
# Cell 4 - Validate locked prereg identity and reviewed upstream smoke notes.
# This cell compares the identity hash stored inside the bundle, not the raw file SHA256.

bundle = read_json(PREREG_BUNDLE)
locked_identity_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', locked_identity_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert locked_identity_hash == EXPECTED_LOCKED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash does not match reviewed hash.'

review_note_a2 = A2_A2B_REVIEWED_RUN / 'phase1_a2_a2b_model_smoke_review_note.json'
review_note_a2d = A2D_REVIEWED_RUN / 'phase1_a2d_riemannian_smoke_review_note.json'
for label, path in {'A2/A2b review note': review_note_a2, 'A2d review note': review_note_a2d}.items():
    print(label, 'exists =', path.exists(), '|', path)
    assert path.exists(), f'Missing reviewed source: {path}'

a2_review = read_json(review_note_a2)
a2d_review = read_json(review_note_a2d)
print('A2/A2b review status:', a2_review.get('status'))
print('A2d review status:', a2d_review.get('status'))
assert a2_review.get('status') == 'phase1_a2_a2b_model_smoke_review_pass_non_claim'
assert a2d_review.get('status') == 'phase1_a2d_riemannian_smoke_review_pass_non_claim'

print('Source-chain review passed. These prior outputs remain non-claim diagnostics only.')

In [ ]:
# Cell 5 - Hash relevant code/config/doc surfaces for the A2c plan.

tracked_sources = [
    REPO_DIR / 'src' / 'phase1' / 'a2c_smoke.py',
    REPO_DIR / 'src' / 'cli.py',
    REPO_DIR / 'configs' / 'phase1' / 'a2c_smoke.json',
    REPO_DIR / 'configs' / 'models' / 'coral.yaml',
    REPO_DIR / 'tests' / 'unit' / 'test_phase1_a2c_smoke.py',
    REPO_DIR / 'docs' / '01_v55_doc_code_crosswalk.md',
    REPO_DIR / 'docs' / '02_colab_local_runbook.md',
    REPO_DIR / 'notebooks' / '10_colab_phase1_a2c_coral_smoke.ipynb',
]
source_hashes = {}
for path in tracked_sources:
    source_hashes[str(path.relative_to(REPO_DIR))] = {
        'exists': path.exists(),
        'sha256': sha256_file(path) if path.exists() else None,
    }
print(json.dumps(source_hashes, indent=2))
assert all(item['exists'] for item in source_hashes.values()), 'A2c source/config/notebook/test surface is incomplete.'

In [ ]:
# Cell 6 - A2c runner preflight.
# mne/signal extras are required only when launching real EDF extraction, not for this manual-hold planning pass.

preflight = {
    'notebook_fix_marker': NOTEBOOK_FIX_MARKER,
    'runner_module_exists': (REPO_DIR / 'src' / 'phase1' / 'a2c_smoke.py').exists(),
    'phase_config_exists': (REPO_DIR / 'configs' / 'phase1' / 'a2c_smoke.json').exists(),
    'coral_config_exists': (REPO_DIR / 'configs' / 'models' / 'coral.yaml').exists(),
    'numpy_available': importlib.util.find_spec('numpy') is not None,
    'mne_available': importlib.util.find_spec('mne') is not None,
}
cli_help = run(['python', '-m', 'src.cli', 'phase1_real', '--help'], cwd=REPO_DIR, check=False, capture=True)
preflight['cli_help_returncode'] = cli_help.returncode
preflight['cli_exposes_a2c_smoke'] = '--a2c-smoke' in (cli_help.stdout or '')
import_check = run(['python', '-c', 'from src.phase1.a2c_smoke import run_phase1_a2c_smoke; print("a2c import ok")'], cwd=REPO_DIR, check=False, capture=True)
preflight['runner_import_returncode'] = import_check.returncode
preflight['runner_import_available'] = import_check.returncode == 0
coral_text = (REPO_DIR / 'configs' / 'models' / 'coral.yaml').read_text(encoding='utf-8') if preflight['coral_config_exists'] else ''
preflight['coral_revision_note_present'] = 'non_claim_smoke_revision_note' in coral_text
preflight['coral_not_placeholder'] = 'placeholder_until_prereg' not in coral_text

blockers = []
if not preflight['runner_module_exists']:
    blockers.append('src/phase1/a2c_smoke.py is missing')
if not preflight['phase_config_exists']:
    blockers.append('configs/phase1/a2c_smoke.json is missing')
if not preflight['cli_exposes_a2c_smoke']:
    blockers.append('CLI does not expose --a2c-smoke')
if not preflight['runner_import_available']:
    blockers.append('A2c runner import failed')
if not preflight['numpy_available']:
    blockers.append('numpy is required for the A2c smoke proxy')
if not preflight['coral_revision_note_present'] or not preflight['coral_not_placeholder']:
    blockers.append('configs/models/coral.yaml does not contain the non-claim smoke revision note')
preflight['blockers'] = blockers
print(json.dumps(preflight, indent=2))

## A2c smoke artifact contract

If the manual hold is later released, the CLI run must write and review these artifacts under `artifacts/phase1_a2c_smoke/<timestamp>/`:

- `phase1_a2c_smoke_inputs.json`
- `phase1_a2c_smoke_summary.json`
- `phase1_a2c_smoke_report.md`
- `a2c_metrics_smoke.json`
- `a2c_domain_audit.json`
- `a2c_coral_penalty_audit.json`
- `a2c_feature_manifest.json`
- `calibration_a2c_smoke_report.json`
- `negative_controls_a2c_smoke_report.json`
- `influence_a2c_smoke_report.json`
- `fold_logs/*.json`
- `a2c_logits_smoke/*.json`

Required interpretation guard:

- `claim_ready` must be false.
- `final_a2c_comparator` must be false.
- `does_not_estimate_privileged_transfer_efficacy` must be true.
- Outer-test subjects must not appear in normalization fit, CORAL statistics fit, beta selection, classifier fit, or calibration fit.

In [ ]:
# Cell 7 - Write the A2c plan artifact.

plan_stamp = utc_stamp()
plan_dir = A2C_PLAN_ROOT / plan_stamp
plan = {
    'status': 'phase1_a2c_coral_smoke_plan_recorded',
    'created_utc': plan_stamp,
    'notebook_fix_marker': NOTEBOOK_FIX_MARKER,
    'locked_prereg_identity_hash': locked_identity_hash,
    'raw_prereg_file_sha256': raw_prereg_file_sha256,
    'prereg_bundle': str(PREREG_BUNDLE),
    'phase1_readiness_run': str(PHASE1_READINESS_RUN),
    'dataset_root': str(DATASET_ROOT),
    'a2_a2b_reviewed_run': str(A2_A2B_REVIEWED_RUN),
    'a2d_reviewed_run': str(A2D_REVIEWED_RUN),
    'a2c_output_root': str(A2C_OUTPUT_ROOT),
    'preflight': preflight,
    'source_hashes': source_hashes,
    'planned_cli': [
        'python', '-m', 'src.cli', 'phase1_real',
        '--profile', 't4_safe',
        '--config', str(PREREG_BUNDLE),
        '--readiness-run', str(PHASE1_READINESS_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--output-root', str(A2C_OUTPUT_ROOT),
        '--a2c-smoke',
        '--phase-config', 'configs/phase1/a2c_smoke.json',
        '--max-outer-folds', '2',
    ],
    'non_claim_debug_only': True,
    'claim_ready': False,
    'launch_policy': 'manual_hold_default_false; explicit acknowledgement required before training',
    'not_ok_to_claim': [
        'decoder_efficacy',
        'A2c_final_comparator_performance',
        'A2c_superiority_or_inferiority_against_other_comparators',
        'privileged_transfer_efficacy',
        'full_phase1_neural_comparator_performance',
    ],
}
write_json(plan_dir / 'phase1_a2c_coral_smoke_plan.json', plan)
(A2C_PLAN_ROOT / 'latest.txt').write_text(str(plan_dir), encoding='utf-8')
print(json.dumps({
    'status': plan['status'],
    'plan_dir': str(plan_dir),
    'runner_available': not bool(preflight['blockers']),
    'blockers': preflight['blockers'],
    'locked_prereg_identity_hash': locked_identity_hash,
}, indent=2))

In [ ]:
# Cell 8 - Manual launch gate.
# First rerun after pull should keep RUN_A2C_CORAL_SMOKE = False.
# To launch later, set INSTALL_SIGNAL_EXTRAS = True in Cell 1, rerun bootstrap/tests, then set the flag and ACK below.

RUN_A2C_CORAL_SMOKE = False
MANUAL_LAUNCH_ACK = ''
REQUIRED_ACK = 'I reviewed the A2c CORAL plan and understand this is non-claim implementation smoke'

launch_record = {
    'created_utc': utc_stamp(),
    'run_flag': RUN_A2C_CORAL_SMOKE,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight['blockers'],
    'mne_available': preflight['mne_available'],
    'plan_dir': str(plan_dir),
}

if preflight['blockers']:
    launch_record['status'] = 'phase1_a2c_coral_smoke_blocked_preflight'
    write_json(plan_dir / 'phase1_a2c_coral_smoke_launch_record.json', launch_record)
    print(json.dumps(launch_record, indent=2))
    raise RuntimeError('A2c runner/config/CLI preflight has blockers. Do not launch.')
elif not RUN_A2C_CORAL_SMOKE:
    launch_record['status'] = 'phase1_a2c_coral_smoke_manual_hold'
    write_json(plan_dir / 'phase1_a2c_coral_smoke_launch_record.json', launch_record)
    print(json.dumps(launch_record, indent=2))
    print('HELD: Runner appears available, but training was not launched because manual flag is False.')
    print('NEXT: review this plan artifact, then only later rerun with signal extras and explicit acknowledgement if appropriate.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch A2c smoke without explicit non-claim acknowledgement.')
elif not preflight['mne_available']:
    raise RuntimeError('A2c real EDF feature extraction requires mne/signal extras. Set INSTALL_SIGNAL_EXTRAS = True in Cell 1, rerun bootstrap/tests, then relaunch intentionally.')
else:
    launch_record['status'] = 'phase1_a2c_coral_smoke_launch_approved'
    write_json(plan_dir / 'phase1_a2c_coral_smoke_launch_record.json', launch_record)
    cmd = plan['planned_cli']
    run(cmd, cwd=REPO_DIR)
    print('A2c CORAL smoke command completed. Review generated artifacts before any further run.')

In [ ]:
# Cell 9 - Closeout summary for the manual-hold pass or later launch pass.

latest = latest_run(A2C_OUTPUT_ROOT)
print('================ Phase 1 A2c CORAL Smoke Closeout ================')
print('Notebook fix marker:', NOTEBOOK_FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', not bool(preflight['blockers']))
print('Run flag:', RUN_A2C_CORAL_SMOKE)
print('Expected locked prereg identity hash:', EXPECTED_LOCKED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', locked_identity_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
print('Blockers:', preflight['blockers'])

if latest and RUN_A2C_CORAL_SMOKE:
    print('Latest A2c output:', latest)
    expected = [
        'phase1_a2c_smoke_inputs.json',
        'phase1_a2c_smoke_summary.json',
        'phase1_a2c_smoke_report.md',
        'a2c_metrics_smoke.json',
        'a2c_domain_audit.json',
        'a2c_coral_penalty_audit.json',
        'a2c_feature_manifest.json',
        'calibration_a2c_smoke_report.json',
        'negative_controls_a2c_smoke_report.json',
        'influence_a2c_smoke_report.json',
    ]
    for name in expected:
        print(name, 'exists =', (latest / name).exists())
    summary_path = latest / 'phase1_a2c_smoke_summary.json'
    if summary_path.exists():
        summary = read_json(summary_path)
        print(json.dumps({
            'status': summary.get('status'),
            'comparator': summary.get('comparator'),
            'n_outer_folds': summary.get('n_outer_folds'),
            'blockers': summary.get('blockers'),
            'claim_ready': summary.get('claim_ready'),
            'final_a2c_comparator': summary.get('final_a2c_comparator'),
            'scientific_limit': summary.get('scientific_limit'),
        }, indent=2))
    print('CHECK REQUIRED: A2c smoke command was launched. Review fold logs, domain audit, CORAL penalty audit and metrics before continuing.')
elif preflight['blockers']:
    print('BLOCKED: A2c runner/config/CLI is unavailable. Fix repo implementation first.')
else:
    print('HELD: Runner appears available, but training was not launched because manual flag is False.')
    print('NEXT: close this first-pass notebook after confirming the plan artifact, then relaunch only with explicit non-claim acknowledgement if appropriate.')

print('\nNOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2c final comparator performance, or privileged-transfer efficacy.')